# 03 — Model Training
**Health Risk Triage AI — Phase 1**

This notebook presents the multi-model comparison results from `src/train.py`. It loads the saved CV results and fitted models, visualises performance across all 5 model families, and analyses what the cross-validation scores tell us about each algorithm's suitability for this clinical triage task.

---
**Pipeline position:** `preprocessing.py` → `models.py` → `train.py` → **[YOU ARE HERE]** → `evaluate.py`

**Inputs:**
- `reports/cv_results.json` — CV scores and best hyperparameters per model
- `models/*_best.joblib` — fitted model artifacts
- `data/processed/phase1/phase1_train.parquet`

**Output:** Observations only — no files written by this notebook

## 0. Imports and Config

In [ ]:
import sys
sys.path.append('..')

import json
import yaml
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

FEATURES   = config['features']['model_features']
TARGET     = config['features']['target']
LABELS     = ['Low', 'Medium', 'High', 'Critical']
COLORS     = ['steelblue', 'mediumseagreen', 'orange', 'tomato']
MODEL_ORDER = ['xgboost', 'random_forest', 'decision_tree', 'mlp', 'logistic_regression']
MODEL_COLORS = ['#E53935', '#43A047', '#1E88E5', '#FB8C00', '#8E24AA']

print('Config loaded.')

## 1. Load CV Results

In [ ]:
with open('../reports/cv_results.json') as f:
    cv_results = json.load(f)

# Build a clean summary DataFrame
rows = []
for name in MODEL_ORDER:
    if name not in cv_results:
        continue
    r = cv_results[name]
    rows.append({
        'Model'         : name.replace('_', ' ').title(),
        'CV F1-Macro'   : r.get('cv_f1_macro_mean'),
        'CV Std'        : r.get('cv_f1_macro_std'),
        'Train Time (s)': r.get('train_time_sec'),
    })

cv_df = pd.DataFrame(rows)
print('Cross-Validation Results Summary:')
print(cv_df.to_string(index=False))

## 2. CV F1-Macro Comparison

Primary metric for model selection. Macro-averaged F1 weights all 4 urgency classes equally — it penalises models that ignore the rare High/Critical classes to maximise accuracy on the dominant Low class.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

names  = cv_df['Model'].tolist()
scores = cv_df['CV F1-Macro'].tolist()
stds   = cv_df['CV Std'].tolist()

bars = ax.barh(names, scores, xerr=stds,
               color=MODEL_COLORS[:len(names)],
               edgecolor='white', capsize=5,
               error_kw={'elinewidth': 1.5, 'ecolor': 'gray'})

ax.set_xlabel('CV F1-Macro (5-fold stratified)', fontsize=11)
ax.set_title('Model Comparison — Cross-Validation F1-Macro\n'
             '(error bars = ± std across 5 folds)', fontweight='bold')
ax.set_xlim(0, 1.05)
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='0.5 baseline')

for bar, score, std in zip(bars, scores, stds):
    ax.text(score + std + 0.01,
            bar.get_y() + bar.get_height()/2,
            f'{score:.4f} ± {std:.4f}',
            va='center', fontsize=9)

ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

best = cv_df.loc[cv_df['CV F1-Macro'].idxmax(), 'Model']
print(f'Best model by CV F1-Macro: {best}')

## 3. Accuracy vs Training Time Trade-off

For a triage tool intended for low-resource deployment, inference speed matters. This plot shows the performance-cost trade-off across model families.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for i, row in cv_df.iterrows():
    ax.scatter(row['Train Time (s)'], row['CV F1-Macro'],
               s=200, color=MODEL_COLORS[i],
               zorder=5, edgecolors='white', linewidth=1.5)
    ax.annotate(row['Model'],
                (row['Train Time (s)'], row['CV F1-Macro']),
                textcoords='offset points', xytext=(8, 4),
                fontsize=9)

ax.set_xlabel('Training Time (seconds, log scale)')
ax.set_ylabel('CV F1-Macro')
ax.set_title('Performance vs Training Time Trade-off', fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Decision Tree is in the top-right quadrant (fast + reasonable accuracy).')
print('XGBoost dominates on accuracy with moderate training time.')
print('SVM was excluded: training time exceeded 2.5 hours (off this chart entirely).')

## 4. Best Hyperparameters Per Model

Selected by RandomizedSearchCV (20 iterations, 5-fold CV, f1_macro scoring).

In [ ]:
print('Best hyperparameters selected by RandomizedSearchCV:\n')
for name in MODEL_ORDER:
    if name not in cv_results:
        continue
    params = cv_results[name].get('best_params', {})
    print(f'  {name.replace("_", " ").title()}')
    if params:
        for k, v in params.items():
            print(f'    {k:<25} : {v}')
    else:
        print('    (default parameters used)')
    print()

## 5. Decision Tree — Rule Path Visualisation

The Decision Tree is the most interpretable model in this comparison. A shallow tree (max_depth ≤ 4) produces rule paths a clinician can read directly — mapping to ETAT-style 'if X do Y' logic.

In [ ]:
from sklearn.tree import export_text, DecisionTreeClassifier

dt_model = joblib.load('../models/decision_tree_best.joblib')

# Print text rule paths (top 4 levels only for readability)
rules = export_text(dt_model,
                    feature_names=FEATURES,
                    max_depth=4)
print('Decision Tree Rule Paths (top 4 levels):')
print('=' * 55)
print(rules[:3000])  # Truncate for display — full tree may be large
print('...')

## 6. Random Forest — Feature Importance (Gini)

Built-in feature importance from tree node impurity reduction. Compare this against SHAP importance in notebook 04 to check consistency.

In [ ]:
rf_model = joblib.load('../models/random_forest_best.joblib')

importances = pd.Series(
    rf_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['tomato' if f == 'NEWS2Score' else 'steelblue' for f in importances.index]
ax.barh(importances.index, importances.values,
        color=colors, edgecolor='white')
ax.set_xlabel('Gini Feature Importance')
ax.set_title('Random Forest — Gini Feature Importance', fontweight='bold')

red_patch  = mpatches.Patch(color='tomato',    label='NEWS2Score (top feature)')
blue_patch = mpatches.Patch(color='steelblue', label='Other features')
ax.legend(handles=[red_patch, blue_patch], fontsize=9)
plt.tight_layout()
plt.show()

print('Top 5 features by Gini importance:')
print(importances.sort_values(ascending=False).head().round(4).to_string())

## 7. XGBoost — Built-in Feature Importance

XGBoost provides three importance types: `weight` (split count), `gain` (improvement per split), `cover` (samples affected). `gain` is most comparable to SHAP and Gini importance.

In [ ]:
xgb_model = joblib.load('../models/xgboost_best.joblib')

importance_types = ['weight', 'gain', 'cover']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, imp_type in zip(axes, importance_types):
    scores = xgb_model.get_booster().get_score(importance_type=imp_type)
    imp_s  = pd.Series(scores).reindex(FEATURES).fillna(0).sort_values(ascending=True)
    colors = ['tomato' if f == 'NEWS2Score' else 'steelblue' for f in imp_s.index]
    ax.barh(imp_s.index, imp_s.values, color=colors, edgecolor='white')
    ax.set_title(f'XGBoost — {imp_type.title()} Importance')
    ax.set_xlabel(imp_type.title())

plt.suptitle('XGBoost Feature Importance (3 methods)', fontweight='bold')
plt.tight_layout()
plt.show()

print('Gain importance is most directly comparable to SHAP (see notebook 04).')
print('NEWS2Score dominance should appear across all three importance types.')

## 8. Model Complexity vs Interpretability

Summary of the interpretability-accuracy trade-off across model families — relevant for clinical deployment decisions.

In [ ]:
summary = [
    {'Model': 'Logistic Regression', 'Family': 'Linear',
     'Interpretability': 5, 'CV F1-Macro': 0.5578,
     'Clinical Deploy': 'Easy — coefficients map to feature weights'},
    {'Model': 'Decision Tree',       'Family': 'Tree',
     'Interpretability': 5, 'CV F1-Macro': 0.8061,
     'Clinical Deploy': 'Easy — rule paths readable by clinician'},
    {'Model': 'Random Forest',       'Family': 'Ensemble (bag)',
     'Interpretability': 3, 'CV F1-Macro': 0.8655,
     'Clinical Deploy': 'Medium — SHAP needed for explanation'},
    {'Model': 'XGBoost',             'Family': 'Ensemble (boost)',
     'Interpretability': 3, 'CV F1-Macro': 0.9413,
     'Clinical Deploy': 'Medium — SHAP needed, best performance'},
    {'Model': 'MLP',                 'Family': 'Neural',
     'Interpretability': 1, 'CV F1-Macro': 0.7807,
     'Clinical Deploy': 'Hard — black box, underperforms ensembles here'},
]

summary_df = pd.DataFrame(summary)
print(summary_df[['Model', 'Family', 'CV F1-Macro',
                   'Interpretability', 'Clinical Deploy']].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    summary_df['Interpretability'],
    summary_df['CV F1-Macro'],
    s=300, c=MODEL_COLORS[:5], zorder=5,
    edgecolors='white', linewidth=1.5
)
for _, row in summary_df.iterrows():
    ax.annotate(row['Model'],
                (row['Interpretability'], row['CV F1-Macro']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Interpretability (5=fully interpretable, 1=black box)')
ax.set_ylabel('CV F1-Macro')
ax.set_title('Interpretability vs Performance Trade-off', fontweight='bold')
ax.set_xlim(0, 6.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nKey finding: XGBoost dominates on performance but needs SHAP for clinical trust.')
print('Decision Tree is competitive (0.81) and fully interpretable — viable for deployment.')

## 9. Literature Comparison

Contextualising results against the EWS ML scoping review (JMIR 2021;23(2):e25187).

In [ ]:
lit_comparison = pd.DataFrame([
    {'Source': 'This project — XGBoost',          'Metric': 'CV F1-Macro', 'Score': 0.9413},
    {'Source': 'Jang et al. — LSTM',              'Metric': 'AUROC',       'Score': 0.933},
    {'Source': 'Kwon et al. — RNN',               'Metric': 'AUROC',       'Score': 0.850},
    {'Source': 'Traditional NEWS vs ML (avg)',     'Metric': 'AUROC',       'Score': 0.780},
    {'Source': 'Kwon et al. — MEWS (baseline)',   'Metric': 'AUROC',       'Score': 0.603},
])

print('Literature Comparison (note: different metrics — directional only):')
print(lit_comparison.to_string(index=False))
print()
print('Caveat: F1-Macro and AUROC are different metrics and not directly comparable.')
print('Both measure model discrimination ability but from different angles.')
print('This comparison is directional — showing our results sit in the same range')
print('as published ML-EWS models, not claiming direct equivalence.')